# Intermediate Quant Research Tutorial: Multi-Asset Strategies

**Learning objectives:**
- Load multiple assets (equities + bonds)
- Build trend-following strategies with Backtrader
- Implement pair trading / relative value strategies
- Run multi-asset backtests

This tutorial demonstrates how to research strategies that span multiple asset classes using Backtrader.


## 1. Setup

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
from pathlib import Path

project_root = Path('/Users/zelin/Desktop/PA Investment/Invest_strategy')
sys.path.insert(0, str(project_root))

import backtrader as bt
from backend.backtest_engine import BacktestEngine, IBKRDataFeed

print("Setup complete")

## 2. Load Multi-Asset Data

In [ ]:
import yfinance as yf

# Load IVV (S&P 500) and TLT (20+ Year Treasury)
tickers = ["IVV", "TLT"]
start_date = "2020-01-01"
end_date = "2024-12-31"

data_dict = {}
for ticker in tickers:
    df = yf.download(ticker, start=start_date, end=end_date, progress=False)
    df = df.droplevel('Ticker', axis=1)
    df = df.reset_index()
    df.columns = ['date', 'open', 'high', 'low', 'close', 'volume']
    data_dict[ticker] = df

print(f"Loaded {len(data_dict['IVV'])} days of data for each asset")
print(f"Tickers: {tickers}")

## 3. Strategy 1: Trend Following (SMA Crossover)

In [ ]:
class SMACrossover(bt.Strategy):
    """
    Trend following using SMA crossover.
    - Long when price > 200-day SMA
    - Short when price < 200-day SMA
    """
    params = (
        ('sma_period', 200),
    )
    
    def __init__(self):
        self.sma = bt.indicators.SimpleMovingAverage(
            self.data.close, 
            period=self.params.sma_period
        )
        self.crossover = bt.indicators.CrossOver(self.data.close, self.sma)
        
    def next(self):
        if len(self) < self.params.sma_period:
            return
            
        if not self.position:
            if self.data.close[0] > self.sma[0]:
                self.buy()
        else:
            if self.data.close[0] < self.sma[0]:
                self.sell()

print("Strategy defined: SMACrossover")

## 4. Run IVV Trend Following Backtest

In [ ]:
# Run backtest on IVV
engine_ivv = BacktestEngine(cash=100000, commission=0.001)
engine_ivv.add_data(IBKRDataFeed(dataname=data_dict['IVV'].copy()), name='IVV')
engine_ivv.add_strategy(SMACrossover)
result_ivv = engine_ivv.run_backtest()

print("=== IVV Trend Following Results ===")
print(f"Final Value: ${result_ivv['final_value']:,.0f}")
print(f"Total Return: {result_ivv['total_return']*100:.2f}%")
print(f"Sharpe Ratio: {result_ivv['sharpe_ratio']:.3f}")
print(f"Max Drawdown: {result_ivv['max_drawdown']*100:.2f}%")

## 5. Run TLT Trend Following Backtest

In [ ]:
# Run backtest on TLT
engine_tlt = BacktestEngine(cash=100000, commission=0.001)
engine_tlt.add_data(IBKRDataFeed(dataname=data_dict['TLT'].copy()), name='TLT')
engine_tlt.add_strategy(SMACrossover)
result_tlt = engine_tlt.run_backtest()

print("=== TLT Trend Following Results ===")
print(f"Final Value: ${result_tlt['final_value']:,.0f}")
print(f"Total Return: {result_tlt['total_return']*100:.2f}%")
print(f"Sharpe Ratio: {result_tlt['sharpe_ratio']:.3f}")
print(f"Max Drawdown: {result_tlt['max_drawdown']*100:.2f}%")

## 6. Strategy 2: Pair Trading (Spread Mean Reversion)

In [ ]:
class PairTradingStrategy(bt.Strategy):
    """
    Pair trading strategy on IVV vs TLT spread.
    - Calculate spread (IVV/TLT ratio)
    - Long spread when below moving average (expect reversion)
    - Short spread when above moving average
    """
    params = (
        ('sma_period', 60),
        ('entry_std', 1.5),
    )
    
    def __init__(self):
        # We need to access both data feeds
        # This strategy will be applied to a combined spread
        self.spread_sma = None
        
    def next(self):
        if len(self) < self.params.sma_period:
            return
            
        # Note: In practice, you'd calculate spread externally
        # This is a simplified version showing the structure
        
print("Strategy defined: PairTradingStrategy")

## 7. Multi-Data Strategy: Dual Asset Trend

In [ ]:
class DualAssetTrend(bt.Strategy):
    """
    Apply trend following to two assets simultaneously.
    - Go long both when both in uptrend
    - Go short both when both in downtrend
    """
    params = (
        ('sma_period', 200),
    )
    
    def __init__(self):
        # Data 0 = IVV, Data 1 = TLT
        self.sma_ivv = bt.indicators.SimpleMovingAverage(
            self.data0.close, period=self.params.sma_period
        )
        self.sma_tlt = bt.indicators.SimpleMovingAverage(
            self.data1.close, period=self.params.sma_period
        )
        
    def next(self):
        if len(self) < self.params.sma_period:
            return
            
        # Get trends
        ivv_trend = 1 if self.data0.close[0] > self.sma_ivv[0] else -1
        tlt_trend = 1 if self.data1.close[0] > self.sma_tlt[0] else -1
        
        # Both in agreement - trade both
        if ivv_trend == tlt_trend:
            if ivv_trend == 1 and not self.position:
                self.buy()  # Long both
            elif ivv_trend == -1 and not self.position:
                self.sell()  # Short both
        else:
            # Disagreement - exit
            if self.position:
                if self.position.size > 0:
                    self.sell()
                else:
                    self.buy()
                    
print("Strategy defined: DualAssetTrend")

In [ ]:
# Run multi-asset backtest
engine_dual = BacktestEngine(cash=100000, commission=0.001)
engine_dual.add_data(IBKRDataFeed(dataname=data_dict['IVV'].copy()), name='IVV')
engine_dual.add_data(IBKRDataFeed(dataname=data_dict['TLT'].copy()), name='TLT')
engine_dual.add_strategy(DualAssetTrend)
result_dual = engine_dual.run_backtest()

print("=== Dual Asset Trend Strategy Results ===")
print(f"Final Value: ${result_dual['final_value']:,.0f}")
print(f"Total Return: {result_dual['total_return']*100:.2f}%")
print(f"Sharpe Ratio: {result_dual['sharpe_ratio']:.3f}")
print(f"Max Drawdown: {result_dual['max_drawdown']*100:.2f}%")

## 8. Compare All Strategies

In [ ]:
# Compare all strategies
comparison = pd.DataFrame({
    'IVV Trend': [
        f"{result_ivv['total_return']*100:.2f}%",
        f"{result_ivv['sharpe_ratio']:.3f}",
        f"{result_ivv['max_drawdown']*100:.2f}%"
    ],
    'TLT Trend': [
        f"{result_tlt['total_return']*100:.2f}%",
        f"{result_tlt['sharpe_ratio']:.3f}",
        f"{result_tlt['max_drawdown']*100:.2f}%"
    ],
    'Dual Asset': [
        f"{result_dual['total_return']*100:.2f}%",
        f"{result_dual['sharpe_ratio']:.3f}",
        f"{result_dual['max_drawdown']*100:.2f}%"
    ]
}, index=['Total Return', 'Sharpe Ratio', 'Max Drawdown'])

print("=== Strategy Comparison ===")
print(comparison)

## Summary

In this tutorial, you learned:
1. **Multi-Asset Data**: How to load and prepare data for multiple assets
2. **Trend Following**: How to build SMA crossover strategies in Backtrader
3. **Multi-Data Strategies**: How to trade multiple assets with single strategy
4. **Strategy Comparison**: How to compare performance across assets

**Key insights:**
- Trend following works differently for equities vs bonds
- Multi-asset strategies can provide diversification
- Same signal applied to different assets produces different results

**Next steps:**
- Try pair trading with explicit spread calculation
- Add more assets (see advanced tutorial)
- Implement volatility-based position sizing
